# 04 — Inferencia en Video

Este notebook usa el modelo entrenado para procesar un video de cámara trampa de Costa Rica y renderizar las detecciones de fauna silvestre.

**¿Qué hace?**
1. Descarga un video del canal [Guanacaste Wildlife Monitoring](https://www.youtube.com/@guanacastewildlifemonitori8194)
2. Procesa cada frame con el modelo YOLOv8n entrenado
3. Renderiza bounding boxes con etiqueta de especie y porcentaje de confianza
4. Exporta el video final con las anotaciones superpuestas

**Prerequisito:** Haber completado `02_training.ipynb` y tener `best.pt` disponible.

**Tiempo estimado:** 10–30 minutos según la duración del video y la GPU disponible.

## 0. Setup e instalación

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Instalar dependencias
!pip install ultralytics yt-dlp opencv-python-headless -q

import cv2
import os
import subprocess
import time
from pathlib import Path

import numpy as np
from ultralytics import YOLO
from IPython.display import HTML, display

# ─── Rutas — ajustar en Colab ─────────────────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector"
BEST_MODEL   = f"{PROJECT_ROOT}/outputs/yolov8n_gtm_v1/weights/best.pt"
VIDEO_DIR    = f"{PROJECT_ROOT}/outputs/videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

print("✅ Setup listo")

✅ Setup listo


## 1. Descargar video de YouTube

Elegí un video del canal [Guanacaste Wildlife Monitoring](https://www.youtube.com/@guanacastewildlifemonitori8194).  
Pegá el URL abajo. Se descargará en 720p (máximo recomendado para no saturar el almacenamiento).

In [ ]:
# ─── PEGÁ AQUÍ EL URL DEL VIDEO QUE ELEGISTE ─────────────────────────────────
# Ejemplo: https://www.youtube.com/watch?v=XXXXXXXXXXX
YOUTUBE_URL = "https://www.youtube.com/watch?v=HcwK6W23Tx0"  # <-- reemplazar

# ─── Configuración del video ──────────────────────────────────────────────────
# Para Colab gratuito (espacio limitado), procesar solo los primeros N segundos
MAX_DURATION_SEC = 120   # primeros 2 minutos (ajustar según tu capacidad)

RAW_VIDEO_PATH = Path(VIDEO_DIR) / "raw_video.mp4"

if RAW_VIDEO_PATH.exists():
    print(f"✅ Video ya descargado: {RAW_VIDEO_PATH}")
else:
    print(f"Descargando video desde: {YOUTUBE_URL}")
    result = subprocess.run([
      "yt-dlp",
      "--cookies", "/content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/www_youtube_com_cookies.txt",
      "--format", "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best",
      "--output", str(RAW_VIDEO_PATH),
      "--no-playlist",
      YOUTUBE_URL,
  ], capture_output=True, text=True)

    if result.returncode == 0:
        size_mb = RAW_VIDEO_PATH.stat().st_size / 1_000_000
        print(f"✅ Descargado: {RAW_VIDEO_PATH} ({size_mb:.1f} MB)")
    else:
        print("❌ Error en descarga:")
        print(result.stderr)
        raise RuntimeError("Descarga fallida")

✅ Video ya descargado: /content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/outputs/videos/raw_video.mp4


## 2. Inspeccionar el video

In [ ]:
cap = cv2.VideoCapture(str(RAW_VIDEO_PATH))

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_sec = total_frames / fps if fps > 0 else 0

cap.release()

print("Información del video:")
print(f"  Resolución  : {width} x {height}")
print(f"  FPS         : {fps:.2f}")
print(f"  Frames      : {total_frames:,}")
print(f"  Duración    : {duration_sec:.1f}s ({duration_sec/60:.1f} min)")

# Calcular frames a procesar
frames_to_process = min(total_frames, int(MAX_DURATION_SEC * fps))
print(f"\n  Frames a procesar: {frames_to_process:,} ({frames_to_process/fps:.0f}s)")

Información del video:
  Resolución  : 1280 x 720
  FPS         : 30.00
  Frames      : 7,490
  Duración    : 249.7s (4.2 min)

  Frames a procesar: 3,600 (120s)


## 3. Cargar modelo

In [ ]:
model = YOLO(BEST_MODEL)
class_names = model.names
print(f"✅ Modelo cargado — {len(class_names)} clases")

✅ Modelo cargado — 12 clases


## 4. Definir colores y función de render

In [ ]:
# Paleta de colores BGR (OpenCV usa BGR, no RGB)
COLORS_BGR = [
    (75,  25, 230),   # azul
    (48, 180, 60),    # verde
    (25, 225, 255),   # amarillo
    (211, 99, 67),    # azul marino
    (60, 130, 245),   # naranja
    (180, 30, 145),   # púrpura
    (212, 188, 66),   # cyan
    (230, 50, 240),   # magenta
    (80, 239, 191),   # lima
    (212, 190, 250),  # rosa claro
    (153, 153, 73),   # teal
    (220, 190, 220),  # lavanda
]

def draw_detections(frame, boxes, class_names, conf_threshold=0.25):
    """
    Dibuja bounding boxes con etiqueta y confianza sobre un frame de OpenCV.
    Replica el estilo estándar de los detectores YOLO.
    """
    if boxes is None or len(boxes) == 0:
        return frame

    for box in boxes:
        conf   = float(box.conf[0])
        if conf < conf_threshold:
            continue
        cls_id = int(box.cls[0])
        name   = class_names.get(cls_id, str(cls_id))
        color  = COLORS_BGR[cls_id % len(COLORS_BGR)]

        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]

        # Bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        # Etiqueta con fondo sólido
        label     = f"{name} {conf:.0%}"
        font      = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.55
        thickness  = 1
        (tw, th), _ = cv2.getTextSize(label, font, font_scale, thickness)
        # Rectángulo de fondo para la etiqueta
        cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(
            frame, label,
            (x1 + 2, y1 - 4),
            font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA
        )
    return frame

print("✅ Función de render lista")

✅ Función de render lista


## 5. Procesar video y renderizar detecciones

⏱️ Tiempo estimado: ~5-15 minutos para 2 minutos de video en GPU T4.

In [ ]:
OUTPUT_VIDEO_PATH = Path(VIDEO_DIR) / "wildlife_detected.mp4"

# Parámetros de inferencia
CONF_THRESHOLD = 0.30   # umbral de confianza para mostrar detecciones
IOU_THRESHOLD  = 0.45   # umbral de NMS

cap = cv2.VideoCapture(str(RAW_VIDEO_PATH))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO_PATH), fourcc, fps, (width, height)
)

frame_idx   = 0
total_dets  = 0
start_time  = time.time()

print(f"Procesando {frames_to_process:,} frames...")
print(f"Umbral de confianza : {CONF_THRESHOLD:.0%}")
print()

try:
    while cap.isOpened() and frame_idx < frames_to_process:
        ret, frame = cap.read()
        if not ret:
            break

        # Inferencia con YOLO
        results = model.predict(
            source  = frame,
            imgsz   = 640,
            conf    = CONF_THRESHOLD,
            iou     = IOU_THRESHOLD,
            verbose = False,
        )

        # Dibujar detecciones sobre el frame
        annotated = draw_detections(frame.copy(), results[0].boxes, class_names, CONF_THRESHOLD)

        # Agregar contador de frames y tiempo en la esquina
        elapsed = frame_idx / fps
        cv2.putText(
            annotated,
            f"Frame {frame_idx:5d} | {elapsed:.1f}s",
            (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA
        )

        writer.write(annotated)

        if results[0].boxes is not None:
            total_dets += len(results[0].boxes)

        frame_idx += 1

        # Log de progreso cada 100 frames
        if frame_idx % 100 == 0:
            elapsed_total = time.time() - start_time
            fps_proc = frame_idx / elapsed_total
            remaining = (frames_to_process - frame_idx) / fps_proc if fps_proc > 0 else 0
            pct = frame_idx / frames_to_process * 100
            print(f"  [{pct:5.1f}%] Frame {frame_idx:5d}/{frames_to_process} "
                  f"| {fps_proc:.1f} fps proc. "
                  f"| ETA: {remaining:.0f}s "
                  f"| Detecciones hasta ahora: {total_dets}")

finally:
    cap.release()
    writer.release()

elapsed_total = time.time() - start_time
print()
print("=" * 55)
print("VIDEO PROCESADO")
print("=" * 55)
print(f"  Frames procesados   : {frame_idx:,}")
print(f"  Duración procesada  : {frame_idx/fps:.1f}s")
print(f"  Total detecciones   : {total_dets:,}")
print(f"  Tiempo de proceso   : {elapsed_total:.1f}s")
print(f"  Velocidad promedio  : {frame_idx/elapsed_total:.1f} fps")
print(f"  Video de salida     : {OUTPUT_VIDEO_PATH}")

Procesando 3,600 frames...
Umbral de confianza : 30%

  [  2.8%] Frame   100/3600 | 27.2 fps proc. | ETA: 129s | Detecciones hasta ahora: 97
  [  5.6%] Frame   200/3600 | 36.3 fps proc. | ETA: 94s | Detecciones hasta ahora: 159
  [  8.3%] Frame   300/3600 | 40.7 fps proc. | ETA: 81s | Detecciones hasta ahora: 196
  [ 11.1%] Frame   400/3600 | 38.9 fps proc. | ETA: 82s | Detecciones hasta ahora: 428
  [ 13.9%] Frame   500/3600 | 39.4 fps proc. | ETA: 79s | Detecciones hasta ahora: 668
  [ 16.7%] Frame   600/3600 | 41.0 fps proc. | ETA: 73s | Detecciones hasta ahora: 919
  [ 19.4%] Frame   700/3600 | 42.3 fps proc. | ETA: 69s | Detecciones hasta ahora: 1149
  [ 22.2%] Frame   800/3600 | 43.9 fps proc. | ETA: 64s | Detecciones hasta ahora: 1180
  [ 25.0%] Frame   900/3600 | 45.1 fps proc. | ETA: 60s | Detecciones hasta ahora: 1283
  [ 27.8%] Frame  1000/3600 | 46.4 fps proc. | ETA: 56s | Detecciones hasta ahora: 1345
  [ 30.6%] Frame  1100/3600 | 45.5 fps proc. | ETA: 55s | Detecciones ha

## 6. Verificar y mostrar muestra del video

In [ ]:
# Verificar que el video de salida existe y tiene tamaño razonable
if OUTPUT_VIDEO_PATH.exists():
    size_mb = OUTPUT_VIDEO_PATH.stat().st_size / 1_000_000
    print(f"✅ Video guardado: {OUTPUT_VIDEO_PATH}")
    print(f"   Tamaño: {size_mb:.1f} MB")
else:
    print("❌ El video de salida no fue creado")

✅ Video guardado: /content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/outputs/videos/wildlife_detected.mp4
   Tamaño: 184.6 MB


In [ ]:
# Extraer frames de muestra para mostrar en el notebook
import matplotlib.pyplot as plt

cap_out = cv2.VideoCapture(str(OUTPUT_VIDEO_PATH))
total_out = int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT))

# Tomar 6 frames distribuidos uniformemente
sample_frame_idxs = [int(total_out * i / 7) for i in range(1, 7)]
sample_frames = []

for idx in sample_frame_idxs:
    cap_out.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap_out.read()
    if ret:
        sample_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

cap_out.release()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, (ax, frm) in enumerate(zip(axes, sample_frames)):
    ax.imshow(frm)
    ax.set_title(f"Frame {sample_frame_idxs[i]}", fontsize=9)
    ax.axis("off")

plt.suptitle("Muestra del video con detecciones renderizadas", fontsize=13)
plt.tight_layout()
plt.savefig(Path(VIDEO_DIR) / "video_sample_frames.png", dpi=100, bbox_inches="tight")
plt.show()
print("✅ Muestra de frames guardada")

## 7. Estadísticas del video procesado

In [ ]:
# Analizar qué especies aparecieron en el video
print("Analizando detecciones en el video completo...")

cap_analysis = cv2.VideoCapture(str(OUTPUT_VIDEO_PATH))
# Re-correr inferencia sobre el video original para recopilar estadísticas
species_detections = {name: 0 for name in class_names.values()}
frame_detection_counts = []

cap_raw = cv2.VideoCapture(str(RAW_VIDEO_PATH))
fi = 0
while cap_raw.isOpened() and fi < frames_to_process:
    ret, frame = cap_raw.read()
    if not ret:
        break
    results = model.predict(frame, conf=CONF_THRESHOLD, verbose=False, imgsz=640)
    n_dets = 0
    if results[0].boxes is not None:
        for box in results[0].boxes:
            name = class_names.get(int(box.cls[0]), "unknown")
            species_detections[name] += 1
            n_dets += 1
    frame_detection_counts.append(n_dets)
    fi += 1

cap_raw.release()

print("\nEspecies detectadas en el video:")
detected = {k: v for k, v in species_detections.items() if v > 0}
if detected:
    for name, cnt in sorted(detected.items(), key=lambda x: -x[1]):
        bar = "█" * (cnt // 3 + 1)
        print(f"  {name:<22} {cnt:4d}  {bar}")
else:
    print("  (Ninguna detección con este umbral — probar reducir CONF_THRESHOLD)")

print(f"\n  Total frames: {fi}")
print(f"  Frames con al menos 1 detección: {sum(1 for c in frame_detection_counts if c > 0)}")